In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

train_df = session.table(
    "DIAMOND_PRICE_ML.ML_WORKFLOW.DIAMONDS_TRAIN"
)

test_df = session.table(
    "DIAMOND_PRICE_ML.ML_WORKFLOW.DIAMONDS_TEST"
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

train_df.show(5)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Bring the Snowflake tables into pandas
train = train_df.to_pandas()
test = test_df.to_pandas()

# Remove split-helper column
train = train.drop(columns=["RN"], errors="ignore")
test = test.drop(columns=["RN"], errors="ignore")

# Features and target
X_train = train.drop(columns=["PRICE"])
y_train = train["PRICE"]

X_test = test.drop(columns=["PRICE"])
y_test = test["PRICE"]

# Categorical and numerical columns
categorical_features = ["CUT", "COLOR", "CLARITY"]
numeric_features = ["CARAT", "DEPTH", "table", "X", "Y", "Z"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# Model 1: Linear Regression
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Model 2: Random Forest
forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
print("Training Linear Regression...")
linear_model.fit(X_train, y_train)

print("Training Random Forest...")
forest_model.fit(X_train, y_train)

# Predictions
linear_pred = linear_model.predict(X_test)
forest_pred = forest_model.predict(X_test)

# Evaluation function
def evaluate_model(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    print(f"\n{name}")
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")

    return mae, rmse, r2

linear_scores = evaluate_model(
    "Linear Regression",
    y_test,
    linear_pred
)

forest_scores = evaluate_model(
    "Random Forest Regressor",
    y_test,
    forest_pred
)

In [ ]:
# Select the best-performing model
best_model = forest_model

print("Best model: Random Forest Regressor")
print(f"Test R²: {forest_scores[2]:.4f}")
print(f"Test RMSE: {forest_scores[1]:.2f}")
print(f"Test MAE: {forest_scores[0]:.2f}")

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="DIAMOND_PRICE_ML",
    schema_name="ML_WORKFLOW"
)

model_ref = registry.log_model(
    best_model,
    model_name="DIAMOND_PRICE_MODEL",
    version_name="V1",
    comment="Random Forest model for diamond price prediction"
)

print("Model registered successfully!")
print(model_ref)